# Convenient experimenation with Tensorflow

In this notebook we learn about some shortcuts or convenience methods for modelling neuronal networks with tensorflow

Get the necessary data files for the next session from https://www.kaggle.com/datasets/zalando-research/fashionmnist


## Preparing the Fashion MNIST-data

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf 
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
print(f"Tensorflow version {tf.__version__}")

#read data with pandas
dataTrain = pd.read_csv('data/fashion-mnist_train.csv')
dataTest = pd.read_csv('data/fashion-mnist_test.csv')

#convert to numpy and initialize variables for training
X_train, y_train = dataTrain.iloc[:, 1:].to_numpy(), dataTrain.iloc[:, 0].to_numpy()
X_test, y_test = dataTest.iloc[:, 1:].to_numpy(), dataTest.iloc[:, 0].to_numpy()

#Define Classnames
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# prepare ground truth as one-hot encoded values
y_one_hot = tf.one_hot(y_train,len(class_names))
y_one_hot_test = tf.one_hot(y_test,len(class_names))


print("Fashion Mnist")
print(f"Training data size: {X_train.shape}")
print(f"Test data size:     {X_test.shape}")
print(f"One Hot encoded:    {y_one_hot.shape}")

# Flatten Layer 

Not always is the data already in the correct format.
So far we always had one-dimensional training samples as a input for NN-Dense-Layers. It can also be multidimensional if the next layers support this.

Suppose we want to *directly* work with images. They typically are 2D. In order to convert data between the 2D Input and the Dense Layer we will add a Flatten Layer

In [ ]:
#Suppose we have the original data not in the original form
X_train_28x28 = X_train.reshape(-1,28,28)
plt.figure(figsize=(1.5, 1.5))
plt.imshow(X_train_28x28[0], cmap="gray_r")


In [ ]:
# converting the test data as well
X_test_28x28 = X_test.reshape(-1,28,28)
plt.figure(figsize=(1.5, 1.5))
plt.imshow(X_test_28x28[0], cmap="gray_r")

In [ ]:
# Modified network including a flatten layer
from tensorflow.keras.layers import Flatten

model = Sequential(name="Model_with_2D_Input")
model.add(Input((28,28)))                             # previously the input was a flat vector of 784 values
model.add(Flatten())                                  # the flatten layer
model.add(Dense(100, activation="sigmoid"))
model.add(Dense(len(class_names), activation="softmax"))
model.compile(optimizer='sgd', loss="categorical_crossentropy", metrics=["accuracy"])
print(model)

In [ ]:
model.summary()

In [ ]:
model.fit(
    X_train_28x28,     # unflattened input (2D image) can be passed
    y_one_hot,
    epochs=10,
    batch_size=100)

print("Evaluation on Test Data")
print(model.evaluate(X_test_28x28, y_one_hot_test))

# Shortcut: Loss Function SparseCategoricalCrossentropy

Categorial Crossentropy is a loss function for one-hot-endoded multinomial labels. We can ommit the step of one hot encoding the labels and let the work to the loss-fuction.

The cost function operating on one hot encoded values looks like this:

$\mathrm{Cost}(h_\theta(x),y\_onehot) = -\sum_{j=1}^K y\_onehot[j] \log\bigl(h_\theta(x)[j]\bigr)$

A modified version operating with categorical encoding looks like that

$\mathrm{Cost}(h_\theta(x),y) = -\log\bigl(h_\theta(x)[y]\bigr)$

This can be done in Tensorflow-Code by swapping "categorical_crossentropy" with "sparse_categorical_crossentropy" and by by using catgorical labels $label \in { 0,1,...,n}$

In [ ]:
# original code
model = Sequential()
model.add(Input((784,)))
model.add(Dense(100, activation="sigmoid"))
model.add(Dense(len(class_names), activation="softmax"))
model.compile(optimizer='sgd', loss="categorical_crossentropy", metrics=["accuracy"]) # one change necessary herer

model.fit(
    X_train,
    y_one_hot,  # one change necessary here
    epochs=10,
    batch_size=100)

print("Evaluation on Test Data")
print(model.evaluate(X_test, y_one_hot_test)) # one change necessary here


<details>
<summary>Click to expand Solution</summary>
<code>
model = Sequential()
model.add(Input((784,)))
model.add(Dense(100, activation="sigmoid"))
model.add(Dense(len(class_names), activation="softmax"))
model.compile(optimizer='sgd', loss="sparse_categorical_crossentropy", metrics=["accuracy"]) # changed to sparse_categorial_crossentropy here

model.fit(
    X_train,
    y_train,  # you can dircetly use y_train here
    epochs=10,
    batch_size=100)

print("Evaluation on Test Data")
print(model.evaluate(X_test, y_test)) # the different loss has also effect on evaluate .. we must change the expected labels here
</code>
</details>

In [ ]:
# print the first two samples
model.predict(X_test[0:2]) # the loss doesnt change the output here, we still get probabilities and need the argmax

In [ ]:
# print it more readable
np.round(model.predict(X_test[0:2]),2)

# Next simplification, Testtrain-split integrated

Often times we make a test/train spllit before we start the training, to have a reproduceable split, that is kept for a series of experiments. The test data is not used during the training run to update the weights. However it makes sense to have an early preview on the relation between test and training score.  Therefor fit offers parameters to set or split test data.

In [ ]:
model = Sequential()
model.add(Input((784,)))
model.add(Dense(100, activation="sigmoid"))
model.add(Dense(len(class_names), activation="softmax"))
model.compile(optimizer='sgd', loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history = model.fit(
    X_train,
    y_train,  
    epochs=10,
    batch_size=100,
    #validation_split=0.3
    validation_data=(X_test, y_test)
)

print("Evaluation on Test Data")
print(model.evaluate(X_test, y_test)) 


# Graphical Vizualization helps to understand the progress 
Fir returns an history object. It stores the data collected during the run

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

# Creating Sequentical models with the constructor

So far we used the method "add" to add layers to a Sequencial-Model. As a shortcut you can just provide a list in the constructor.

In [ ]:
model = Sequential([
      Dense(100, activation='sigmoid'),
      Dense(10, activation="softmax")
      ])
model.compile(optimizer='sgd', loss="categorical_crossentropy", metrics=['accuracy'])
model.fit(
    X_train,
    y_one_hot,
    epochs=10,
    batch_size=100)